# Chapter 5: Sub-Word Tokenization

Transformer models cannot read raw text. They read a sequence of integer token
IDs, and every ID is a row in a fixed-size embedding matrix. This raises a
practical question: how do we turn arbitrary text, including words the model has
never seen, into a short sequence of IDs drawn from a vocabulary of fixed size?

**Sub-word tokenization** is the answer every modern language model uses. Instead
of one token per word (a vocabulary that grows without bound and breaks on unseen
words), text is split into smaller pieces from a compact, data-driven vocabulary of
a predetermined size (typically 30k to 250k tokens). Any word, including a brand
name, a typo, or a word from another language, decomposes into known sub-word
pieces. A frequent word such as "search" stays a single token; a rare word such as
"antidisestablishmentarianism" splits into several.

This notebook is optional background for the curious. It has two parts:

1. **Byte Pair Encoding from scratch.** We implement the BPE merge algorithm in
   plain Python and watch it build a vocabulary on a tiny corpus, reproducing the
   worked example from book section 5.3, then run it on a larger corpus to see words
   grow into whole tokens.
2. **Real pretrained tokenizers.** We load the tokenizers of BERT, GPT-2, and
   XLM-RoBERTa (configuration and vocabulary files only, no model weights) and
   compare how the three algorithms (WordPiece, BPE, SentencePiece) split the same
   text, mark word boundaries, and handle rare words.

## Setup

We only need the standard library and `shared.display` for part 1. For part 2 we
load tokenizers through HuggingFace `transformers`. We silence the library's
informational warnings up front (it will note that PyTorch is absent, which is fine:
we never touch model weights, only the tokenizers).

In [1]:
import os
import warnings
import logging
from collections import Counter

os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

from shared.display import print_table, display_md

## Part 1: Byte Pair Encoding from Scratch

BPE builds its vocabulary iteratively:

1. Start with every word split into individual characters.
2. Count the frequency of all adjacent pairs of symbols across the corpus.
3. Merge the single most frequent pair into one new symbol.
4. Replace every occurrence of that pair with the new symbol.
5. Repeat from step 2 until the vocabulary reaches the target size.

In the first round the pairs are character pairs. In later rounds a "pair" can join
symbols that earlier merges produced, so tokens grow from characters to syllables to
whole words.

### The algorithm in four short functions

Each word is stored as a list of symbols together with how often the word occurs in
the corpus. Pair counts are weighted by that word frequency, so a pair inside a
common word counts more than the same pair inside a rare one.

In [2]:
def init_words(corpus):
    """Split the corpus into words, each represented as a list of characters."""
    freqs = Counter(corpus.lower().split())
    return [(list(word), freq) for word, freq in freqs.items()]

def pair_stats(words):
    """Count adjacent symbol pairs and record the order of first appearance."""
    counts = Counter()
    first_seen = {}
    for symbols, freq in words:
        for pair in zip(symbols, symbols[1:]):
            if pair not in first_seen:
                first_seen[pair] = len(first_seen)
            counts[pair] += freq
    return counts, first_seen

When several pairs tie for the top count, we break the tie by earliest appearance
when scanning the corpus left to right. This is a simple, deterministic rule, and it
reproduces exactly the merge order of the book example.

In [3]:
def best_pair(counts, first_seen):
    """Most frequent pair; ties broken by earliest appearance in the corpus."""
    return min(counts, key=lambda p: (-counts[p], first_seen[p]))

def merge_pair(words, pair):
    """Replace every occurrence of `pair` with a single joined symbol."""
    a, b = pair
    joined = a + b
    new_words = []
    for symbols, freq in words:
        merged, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                merged.append(joined)
                i += 2
            else:
                merged.append(symbols[i])
                i += 1
        new_words.append((merged, freq))
    return new_words

The training loop merges greedily. For this toy run we stop as soon as no pair
occurs more than once, which is where the book stops; a production run would instead
keep merging until it reaches the target vocabulary size.

In [4]:
def train_bpe(corpus, max_merges=1000, min_count=2):
    """Run BPE and return the final word representations and the merge list."""
    words = init_words(corpus)
    merges = []
    for _ in range(max_merges):
        counts, first_seen = pair_stats(words)
        if not counts:
            break
        pair = best_pair(counts, first_seen)
        if counts[pair] < min_count:
            break
        merges.append((pair, counts[pair]))
        words = merge_pair(words, pair)
    return words, merges

### The book's toy example

We run the exact sentence from book section 5.3: "this course is about this topic".
The word "this" appears twice, which drives the first merges.

In [5]:
toy_corpus = "this course is about this topic"
toy_words, toy_merges = train_bpe(toy_corpus)

print_table(
    [[i + 1, f"{a} + {b}", count, a + b]
     for i, ((a, b), count) in enumerate(toy_merges)],
    headers=["Round", "Most frequent pair", "Count", "New token"],
)

|   Round | Most frequent pair   |   Count | New token   |
|--------:|:---------------------|--------:|:------------|
|       1 | i + s                |       3 | is          |
|       2 | t + h                |       2 | th          |
|       3 | th + is              |       2 | this        |
|       4 | o + u                |       2 | ou          |

The four merges match the book precisely: `i, s` becomes `is` (it occurs three
times, twice in "this" and once in "is"); then `t, h` becomes `th`; then `th, is`
becomes the whole word `this`; finally `o, u` becomes `ou` (shared by "course" and
"about"). After these four merges every remaining adjacent pair occurs only once, so
the toy run stops.

Watch a single word grow into one token as the merges apply in sequence:

In [6]:
display_md(
    "**How \"this\" collapses into a single token:**\n\n"
    "| After round | Pieces of \"this\" |\n"
    "|---|---|\n"
    "| start | t , h , i , s |\n"
    "| 1 (merge is) | t , h , is |\n"
    "| 2 (merge th) | th , is |\n"
    "| 3 (merge this) | this |"
)

**How "this" collapses into a single token:**

| After round | Pieces of "this" |
|---|---|
| start | t , h , i , s |
| 1 (merge is) | t , h , is |
| 2 (merge th) | th , is |
| 3 (merge this) | this |

Now we apply the learned merges to the whole sentence. Only "this" and "is" come out
as whole tokens, because the merges actually produced them. "course", "about", and
"topic" stay split into the pieces that survive: the only merge reaching them is
`ou`.

In [7]:
# "".join(symbols) rebuilds the original word, so we can map each word to its pieces.
encoding = {"".join(sym): sym for sym, freq in toy_words}

sentence_tokens = toy_corpus.split()
print_table(
    [[w, " · ".join(encoding[w]), "yes" if len(encoding[w]) == 1 else "no"]
     for w in dict.fromkeys(sentence_tokens)],
    headers=["Word", "Sub-word pieces", "Whole token?"],
)

display_md(
    "**Encoded sentence:** "
    + "  |  ".join(" · ".join(encoding[w]) for w in sentence_tokens)
)

| Word   | Sub-word pieces    | Whole token?   |
|:-------|:-------------------|:---------------|
| this   | this               | yes            |
| course | c · ou · r · s · e | no             |
| is     | is                 | yes            |
| about  | a · b · ou · t     | no             |
| topic  | t · o · p · i · c  | no             |

**Encoded sentence:** this  |  c · ou · r · s · e  |  is  |  a · b · ou · t  |  this  |  t · o · p · i · c

This is the central lesson of BPE: the vocabulary is just the list of merges plus the
starting characters. A word is a whole token only if the training corpus made it
frequent enough to be merged all the way; everything else falls back to pieces.

### A larger corpus: watching words become tokens

With only a handful of words, BPE stops almost immediately. On a larger corpus with
repeated vocabulary, the merges build up longer and longer sub-words, and frequent
words eventually become single tokens. Here the words "token", "tokens", "encoding",
and "search" repeat often enough to be worth merging.

In [8]:
big_corpus = (
    "the tokenizer learns tokens from the corpus "
    "the tokenizer splits tokens into sub word tokens "
    "sub word tokenization builds a fixed vocabulary of tokens "
    "the encoder encodes tokens and the decoder decodes tokens "
    "search engines index tokens and search ranks tokens "
    "tokens tokens tokens encoding encoding encoding search search search"
)

big_words, big_merges = train_bpe(big_corpus, max_merges=40)

display_md(
    f"Ran **{len(big_merges)} merges** on a corpus of "
    f"**{len(big_corpus.split())} words** "
    f"(**{len(set(big_corpus.split()))} distinct**). "
    f"The starting alphabet had **{len(set(big_corpus.replace(' ', '')))} characters**; "
    f"each merge adds one new entry to the vocabulary."
)

Ran **33 merges** on a corpus of **50 words** (**26 distinct**). The starting alphabet had **24 characters**; each merge adds one new entry to the vocabulary.

The first merges below join the most common character pairs; later ones stitch those
into recognisable word stems. The exact sequence depends entirely on the corpus
frequencies.

In [9]:
print_table(
    [[i + 1, f"{a} · {b}", count, a + b]
     for i, ((a, b), count) in enumerate(big_merges[:12])],
    headers=["Round", "Pair merged", "Count", "New token"],
)

|   Round | Pair merged   |   Count | New token   |
|--------:|:--------------|--------:|:------------|
|       1 | e · n         |      20 | en          |
|       2 | t · o         |      15 | to          |
|       3 | to · k        |      14 | tok         |
|       4 | tok · en      |      14 | token       |
|       5 | token · s     |      11 | tokens      |
|       6 | c · o         |       8 | co          |
|       7 | a · r         |       7 | ar          |
|       8 | co · d        |       7 | cod         |
|       9 | e · ar        |       6 | ear         |
|      10 | i · n         |       6 | in          |
|      11 | t · h         |       5 | th          |
|      12 | th · e        |       5 | the         |

Now encode a few words. Frequent words collapse toward a single token, while a word
the corpus never emphasised, or a word absent from the corpus entirely, stays broken
into small pieces. That is exactly how a real tokenizer handles an unseen word.

In [10]:
big_encoding = {"".join(sym): sym for sym, freq in big_words}

def encode_word(word):
    """Apply the trained merges to a single (possibly unseen) word."""
    if word in big_encoding:
        return big_encoding[word]
    symbols = list(word)
    for (a, b), _ in big_merges:
        merged, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                merged.append(a + b)
                i += 2
            else:
                merged.append(symbols[i])
                i += 1
        symbols = merged
    return symbols

samples = ["tokens", "tokenizer", "search", "encoding", "retrieval", "xylophone"]
print_table(
    [[w, " · ".join(encode_word(w)), len(encode_word(w)),
      "in corpus" if w in big_encoding else "unseen"]
     for w in samples],
    headers=["Word", "Pieces", "# pieces", "Source"],
)

| Word      | Pieces                            |   # pieces | Source    |
|:----------|:----------------------------------|-----------:|:----------|
| tokens    | tokens                            |          1 | in corpus |
| tokenizer | tokenizer                         |          1 | in corpus |
| search    | search                            |          1 | in corpus |
| encoding  | encoding                          |          1 | in corpus |
| retrieval | r · e · t · r · i · e · v · a · l |          9 | unseen    |
| xylophone | x · y · l · o · p · h · o · n · e |          9 | unseen    |

Frequent words such as "tokens" and "search" become one or two pieces; "retrieval"
and "xylophone", which the corpus never rewards with merges, stay fragmented into the
sub-words that happen to exist. No word is ever out of vocabulary: in the worst case
it decomposes all the way down to single characters.

## Part 2: Real Pretrained Tokenizers

The book compares three sub-word algorithms. Each is the tokenizer of a widely used
model, and HuggingFace `transformers` lets us load just the tokenizer (small
configuration and vocabulary files, a few megabytes) without downloading the
multi-gigabyte model weights.

| Algorithm | Model we load | Vocab size | Merge criterion |
|---|---|---|---|
| WordPiece | BERT (bert-base-uncased) | 30k | PMI (association strength) |
| BPE (byte-level) | GPT-2 (gpt2) | 50k | Frequency (most common pair) |
| SentencePiece (Unigram) | XLM-RoBERTa (xlm-roberta-base) | 250k | Likelihood (corpus probability) |

In [11]:
from transformers import AutoTokenizer

TOKENIZERS = {
    "BERT (WordPiece)": AutoTokenizer.from_pretrained("bert-base-uncased"),
    "GPT-2 (BPE)": AutoTokenizer.from_pretrained("gpt2"),
    "XLM-RoBERTa (SentencePiece)": AutoTokenizer.from_pretrained("xlm-roberta-base"),
}

display_md("Loaded " + ", ".join(TOKENIZERS) + ".")

Loaded BERT (WordPiece), GPT-2 (BPE), XLM-RoBERTa (SentencePiece).

### Splitting the same sentence three ways

We tokenize one sentence with each tokenizer and show the pieces side by side. The
differences are immediate: BERT lowercases everything (it is the "uncased" model),
the three vocabularies cut the words at different points, and each uses a different
marker to signal where words begin.

In [12]:
sentence = "Semantic search uses transformer embeddings."

rows = []
for name, tok in TOKENIZERS.items():
    pieces = tok.tokenize(sentence)
    rows.append([name, len(pieces), " ".join(pieces)])

print_table(rows, headers=["Tokenizer", "# pieces", "Sub-word pieces"])

| Tokenizer                   |   # pieces | Sub-word pieces                                           |
|:----------------------------|-----------:|:----------------------------------------------------------|
| BERT (WordPiece)            |         10 | semantic search uses transform ##er em ##bed ##ding ##s . |
| GPT-2 (BPE)                 |          9 | Sem antic Ġsearch Ġuses Ġtransformer Ġembed d ings .      |
| XLM-RoBERTa (SentencePiece) |         12 | ▁Sem antic ▁search ▁use s ▁transform er ▁ embe dding s .  |

### Reading the continuation markers

Each algorithm marks word boundaries with a special character. Without it, you could
not tell "to ken" (two words) from "to" plus the continuation "##ken" (one word split
in two). The markers differ because the algorithms treat whitespace differently.

| Tokenizer | Marker | Where it sits | Meaning |
|---|---|---|---|
| BERT (WordPiece) | `##` | prefix on a piece | this piece continues the previous word |
| GPT-2 (BPE) | `Ġ` (shown as Ġ) | prefix on a piece | this piece starts a new word (encodes a leading space) |
| XLM-RoBERTa (SentencePiece) | `▁` (shown as ▁) | prefix on a piece | this piece starts a new word (the SentencePiece space symbol) |

BERT marks the pieces that continue a word; GPT-2 and SentencePiece instead mark the
pieces that start a word. Both conventions carry the same information: given the
pieces and the markers, you can always reconstruct the original spacing.

### Special tokens and token IDs

Beyond the text pieces, most tokenizers add framing tokens that the model relies on.
BERT wraps the input in `[CLS]` and `[SEP]`; XLM-RoBERTa uses `<s>` and `</s>`; GPT-2
adds no framing tokens by default (it was trained for plain left-to-right
generation). The `[CLS]` position is where BERT reads out a sentence-level embedding.

In [13]:
rows = []
for name, tok in TOKENIZERS.items():
    with_special = tok.convert_ids_to_tokens(tok(sentence)["input_ids"])
    rows.append([name, ", ".join(tok.all_special_tokens[:6]),
                 " ".join(with_special)])

print_table(rows, headers=["Tokenizer", "Declared special tokens", "Encoded with framing"])

| Tokenizer                   | Declared special tokens            | Encoded with framing                                                  |
|:----------------------------|:-----------------------------------|:----------------------------------------------------------------------|
| BERT (WordPiece)            | [UNK], [SEP], [PAD], [CLS], [MASK] | [CLS] semantic search uses transform ##er em ##bed ##ding ##s . [SEP] |
| GPT-2 (BPE)                 | <|endoftext|>                      | Sem antic Ġsearch Ġuses Ġtransformer Ġembed d ings .                  |
| XLM-RoBERTa (SentencePiece) | <s>, </s>, <unk>, <pad>, <mask>    | <s> ▁Sem antic ▁search ▁use s ▁transform er ▁ embe dding s . </s>     |

Downstream, every piece is just an integer. Here are the token IDs BERT produces for
the sentence, the actual input a transformer receives. Each ID selects one row of the
embedding matrix.

In [14]:
bert = TOKENIZERS["BERT (WordPiece)"]
enc = bert(sentence)
pieces = bert.convert_ids_to_tokens(enc["input_ids"])

print_table(
    [[i, piece, token_id] for i, (piece, token_id) in enumerate(zip(pieces, enc["input_ids"]))],
    headers=["Position", "Piece", "Token ID"],
)

|   Position | Piece     |   Token ID |
|-----------:|:----------|-----------:|
|          0 | [CLS]     |        101 |
|          1 | semantic  |      21641 |
|          2 | search    |       3945 |
|          3 | uses      |       3594 |
|          4 | transform |      10938 |
|          5 | ##er      |       2121 |
|          6 | em        |       7861 |
|          7 | ##bed     |       8270 |
|          8 | ##ding    |       4667 |
|          9 | ##s       |       2015 |
|         10 | .         |       1012 |
|         11 | [SEP]     |        102 |

### The out-of-vocabulary point: rare and invented words

This is where sub-word tokenization earns its keep. A long rare word and a completely
made-up word have no entry in any vocabulary, yet every tokenizer encodes them without
failing, by falling back to shorter pieces. The number of pieces is a rough measure of
how "surprising" the word is to that tokenizer.

In [15]:
hard_words = ["antidisestablishmentarianism", "cryptofluffernutter", "tokenization"]

rows = []
for word in hard_words:
    for name, tok in TOKENIZERS.items():
        pieces = tok.tokenize(word)
        rows.append([word, name, len(pieces), " ".join(pieces)])

print_table(rows, headers=["Word", "Tokenizer", "# pieces", "Sub-word pieces"])

| Word                         | Tokenizer                   |   # pieces | Sub-word pieces                                   |
|:-----------------------------|:----------------------------|-----------:|:--------------------------------------------------|
| antidisestablishmentarianism | BERT (WordPiece)            |          8 | anti ##dis ##est ##ab ##lish ##ment ##arian ##ism |
| antidisestablishmentarianism | GPT-2 (BPE)                 |          5 | ant idis establishment arian ism                  |
| antidisestablishmentarianism | XLM-RoBERTa (SentencePiece) |          8 | ▁anti d isesta bli shme n tarian ism              |
| cryptofluffernutter          | BERT (WordPiece)            |          6 | crypt ##of ##lu ##ffer ##nut ##ter                |
| cryptofluffernutter          | GPT-2 (BPE)                 |          6 | crypt of l uff ern utter                          |
| cryptofluffernutter          | XLM-RoBERTa (SentencePiece) |          5 | ▁crypto flu ffer nu tter                          |
| tokenization                 | BERT (WordPiece)            |          2 | token ##ization                                   |
| tokenization                 | GPT-2 (BPE)                 |          2 | token ization                                     |
| tokenization                 | XLM-RoBERTa (SentencePiece) |          3 | ▁to ken ization                                   |

The dictionary word "tokenization" is familiar enough that each tokenizer keeps it
short. The invented "cryptofluffernutter" has never appeared in any training corpus,
so it shatters into many small pieces, but it is still encodable: there is no "unknown
word" error, which is exactly the problem sub-word tokenization set out to solve.

### Vocabulary sizes, live

The book lists approximate vocabulary sizes. We can read the real numbers straight
from the loaded tokenizers. Larger vocabularies represent more words as single tokens
(shorter sequences, but a bigger embedding matrix); smaller ones decompose more
aggressively.

In [16]:
CRITERION = {
    "BERT (WordPiece)": "PMI (association strength)",
    "GPT-2 (BPE)": "Frequency (most common pair)",
    "XLM-RoBERTa (SentencePiece)": "Likelihood (corpus probability)",
}

print_table(
    [[name, f"{tok.vocab_size:,}", CRITERION[name]] for name, tok in TOKENIZERS.items()],
    headers=["Tokenizer", "Vocabulary size", "Merge criterion"],
)

| Tokenizer                   |   Vocabulary size | Merge criterion                 |
|:----------------------------|------------------:|:--------------------------------|
| BERT (WordPiece)            |            30,522 | PMI (association strength)      |
| GPT-2 (BPE)                 |            50,257 | Frequency (most common pair)    |
| XLM-RoBERTa (SentencePiece) |           250,002 | Likelihood (corpus probability) |

<div style="border-left: 4px solid #C8102E; background: rgba(200, 16, 46, 0.06); padding: 0.6em 0.9em; margin: 0.6em 0; border-radius: 4px;">
<strong style="color:#C8102E; text-transform:uppercase; font-size:0.78em; letter-spacing:0.06em;">One vocabulary, any text</strong><br>
Sub-word tokenization fixes the vocabulary size (30k to 250k) independently of the corpus, yet encodes any input without an out-of-vocabulary failure. Frequent words stay whole; rare words split into pieces; invented words and other scripts still decompose into something the model can embed. This is what lets one transformer handle a vocabulary it was never explicitly shown.
</div>

## Summary

Both parts tell the same story from two directions. The from-scratch BPE showed that a
vocabulary is nothing more than a starting alphabet plus an ordered list of merges, and
that a word becomes a single token only when the corpus made it frequent enough to be
merged all the way. The pretrained tokenizers showed the three production variants in
action: WordPiece (BERT) marking word-internal pieces with `##`, byte-level BPE (GPT-2)
marking word starts with `Ġ`, and SentencePiece Unigram (XLM-RoBERTa) marking them with
`▁`. All three share the key property: a fixed vocabulary size, decoupled from the model
architecture, that can still encode anything.

| Aspect | BPE | WordPiece | SentencePiece (Unigram) |
|---|---|---|---|
| Builds vocabulary by | merging frequent pairs | merging high-PMI pairs | pruning a large candidate set |
| Word boundary marker | `Ġ` (byte-level) | `##` on continuations | `▁` on word starts |
| Pre-tokenizes on spaces | yes | yes | no (operates on raw bytes) |
| Used by | GPT, Mistral, Qwen | BERT, MiniLM | XLM-RoBERTa, BGE-M3 |

## Try It Yourself

The pieces are all plain functions and loaded tokenizers. Change the inputs and re-run.

**1. Tokenize your own text.** Pass any sentence and compare the three tokenizers.

In [17]:
def compare(text):
    print_table(
        [[name, len(tok.tokenize(text)), " ".join(tok.tokenize(text))]
         for name, tok in TOKENIZERS.items()],
        headers=["Tokenizer", "# pieces", "Pieces"],
    )

compare("BERT lowercases text; GPT-2 keeps case and digits like 2017.")

| Tokenizer                   |   # pieces | Pieces                                                                         |
|:----------------------------|-----------:|:-------------------------------------------------------------------------------|
| BERT (WordPiece)            |         17 | bert lower ##case ##s text ; gp ##t - 2 keeps case and digits like 2017 .      |
| GPT-2 (BPE)                 |         17 | BER T Ġlower cases Ġtext ; ĠG PT - 2 Ġkeeps Ġcase Ġand Ġdigits Ġlike Ġ2017 .   |
| XLM-RoBERTa (SentencePiece) |         18 | ▁BER T ▁lower case s ▁text ; ▁G PT -2 ▁keep s ▁case ▁and ▁digit s ▁like ▁2017. |

**2. Probe byte-level robustness.** Byte-level BPE and SentencePiece can encode any
Unicode, including accents, other scripts, and emoji. Try "café", "naïve", "北京", or
"search is fun ☕" and see how each tokenizer copes.

In [18]:
compare("café in 北京 ☕")

| Tokenizer                   |   # pieces | Pieces                        |
|:----------------------------|-----------:|:------------------------------|
| BERT (WordPiece)            |          5 | cafe in 北 京 [UNK]           |
| GPT-2 (BPE)                 |         11 | c af Ã© Ġin Ġå Į Ĺ äº ¬ Ġâĺ ķ |
| XLM-RoBERTa (SentencePiece) |          6 | ▁café ▁in ▁ 北京 ▁ ☕         |

**3. Retrain your own BPE.** Feed `train_bpe` a different corpus (paste a paragraph of
your choosing) and inspect `merges` to see which sub-words the frequencies reward.

**4. Watch sequence length grow.** For a fixed sentence, count the pieces each
tokenizer produces. A larger vocabulary (XLM-RoBERTa, 250k) usually yields fewer, longer
tokens than a smaller one (BERT, 30k). Longer token sequences cost more compute in the
transformer, which is one reason vocabulary size is a real design trade-off.
```